In [3]:
!pip install "accelerate>=1.1.0"

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
!pip install ipywidgets

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
pip install transformers datasets torch

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import pandas as pd
import torch

## 1. Loading Dataset

In [5]:
train_df = pd.read_csv("../data/train_processed.csv")
val_df = pd.read_csv("../data/val_processed.csv")
test_df = pd.read_csv("../data/test_processed.csv")

### using only i/p and o/p column 

In [6]:
train_df = train_df[["text", "label"]]
val_df = val_df[["text", "label"]]
test_df = test_df[["text", "label"]]

### 2. Convert pandas to hugging-face dataframes

In [7]:
from datasets import Dataset

In [8]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

### 3. Loading Tokenizer

In [9]:
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification

In [10]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

### 4. Tokenizer Function

In [11]:
def tokenize(example):
    return tokenizer(example["text"], truncation=True, padding="max_length", max_length=256)

### 5. Applying tokenization

In [12]:
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/79994 [00:00<?, ? examples/s]

Map:   0%|          | 0/19999 [00:00<?, ? examples/s]

Map:   0%|          | 0/399976 [00:00<?, ? examples/s]

### 6. PyTorch formatting 

In [13]:
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

### 7. Loading Pretrained Model

In [14]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### 8. Training Models

In [15]:
from transformers import TrainingArguments 

In [16]:
training_args = TrainingArguments(
    output_dir="../models/distilbert",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    eval_strategy="epoch",   
    save_strategy="epoch"
)

### 9. Metrics

In [17]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {"accuracy": accuracy_score(labels, preds)}

### 10. Trainer

In [18]:
from transformers import Trainer

In [19]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

### 11. Train

In [28]:
trainer.train()

AttributeError: `AcceleratorState` object has no attribute `distributed_type`. This happens if `AcceleratorState._reset_state()` was called and an `Accelerator` or `PartialState` was not reinitialized.

### 12. Evaluation

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [ ]:
trainer.evaluate(test_dataset)

In [20]:
train_dataset = train_dataset.shuffle(seed=42).select(range(20000))
val_dataset = val_dataset.shuffle(seed=42).select(range(5000))

In [21]:
num_train_epochs=1

In [22]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128   # 🔥 reduce this
    )

In [23]:
per_device_train_batch_size=32

In [24]:
logging_strategy="no"

In [25]:
max_steps=2000

In [26]:
training_args = TrainingArguments(
    output_dir="../models/distilbert",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=1,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="no"
)

In [27]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

False


AssertionError: Torch not compiled with CUDA enabled

In [29]:
import transformers
import accelerate
import torch

print(transformers.__version__)
print(accelerate.__version__)
print(torch.__version__)

5.7.0
1.13.0
2.11.0+cpu
